# Model Response Comparison

Compare the sampling behaviour of models (teacher vs ablated teacher, teacher vs students, or any combination) for specific (cue, color1, color2) inputs.

**Key advantage over the CLI script:** the teacher model is loaded only once and reused for all ablation directions.

---

**Modes available in this notebook:**

| Mode | What to set |
|---|---|
| Healthy vs all ablation directions | `ABLATION_DIRECTIONS = list(range(14))`, leave `STUDENT_RUNS = []` |
| Healthy vs a subset of directions | `ABLATION_DIRECTIONS = [0, 3, 7]` |
| Teacher vs students | `ABLATION_DIRECTIONS = []`, fill `STUDENT_RUNS` |
| Mixed | fill both |

In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.cm as cmx
import matplotlib.colors as mcolors
import pandas as pd
from tqdm.auto import tqdm

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

REPO_ROOT = Path("/scratch3/shaiq_home/repos/behaviour_ddpm")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Import helpers from the script so we don't duplicate logic
from compare_model_responses import (
    load_model_from_run,
    build_trial_info,
    generate_samples_for_model,
    compute_sw_scores,
    plot_single_trial,
    generate_sweep_trials,
    _make_diffusion_cmap,
    _samples_to_2d,
    sliced_wasserstein_distance,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Configuration — edit this cell

In [ ]:
# ── Teacher model ─────────────────────────────────────────────────────────────
TEACHER_RUN = "index_cued_first_diffusion_0.3_swap_7"

# ── Ablation directions to compare against the healthy teacher ─────────────────
# Set to list(range(14)) for all directions, [] to skip, or a subset e.g. [0, 3, 7]
ABLATION_DIRECTIONS = list(range(14))

# ── Extra student runs (trained models, no ablation) ──────────────────────────
# Each entry is a run name (relative to results_link_sampler/) or absolute path.
# Set to [] to skip.
STUDENT_RUNS = []
# Example: STUDENT_RUNS = ["index_cued_first_diffusion_0.3_swap_recovery_0"]

# ── Single-trial input ────────────────────────────────────────────────────────
CUE        = 1       # 1 or 2
COLOR1_DEG = 0.0     # degrees (0–359)
COLOR2_DEG = 90.0    # degrees (0–359)

# ── Sampling & metric settings ────────────────────────────────────────────────
NUM_SAMPLES      = 512
N_SW_PROJECTIONS = 128
SEED             = 42

# ── Output directory for saved plots ─────────────────────────────────────────
OUT_DIR = REPO_ROOT / "results" / "healthy_vs_ablations_notebook"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Load teacher model (run once)

In [ ]:
teacher_model, task, _ = load_model_from_run(TEACHER_RUN, device=device)
teacher_model.eval()

n_dirs = teacher_model.behaviour_nullspace.shape[0]
print(f"Teacher: {TEACHER_RUN}")
print(f"  behaviour_nullspace: {teacher_model.behaviour_nullspace.shape}  ({n_dirs} ablation directions available)")
print(f"  T (diffusion steps): {teacher_model.T}")

if ABLATION_DIRECTIONS:
    bad = [d for d in ABLATION_DIRECTIONS if d >= n_dirs]
    if bad:
        raise ValueError(f"Ablation directions out of range (max {n_dirs-1}): {bad}")

## Build model list

This cell assembles `all_labels`, `all_models`, and `all_ablation_vectors` — one entry per comparison model.  
Re-run this cell (without reloading the teacher) whenever you change `ABLATION_DIRECTIONS` or `STUDENT_RUNS`.

In [ ]:
def _normalised_nullspace(model, direction_idx):
    v = model.behaviour_nullspace[direction_idx].clone()
    return v / (v.norm() + 1e-12)

# Always start with the healthy (unablated) teacher as reference (index 0)
all_labels          = ["Healthy"]
all_models          = [teacher_model]
all_ablation_vectors = [None]

# Add each ablation direction — same model instance, different vector
for d in ABLATION_DIRECTIONS:
    all_labels.append(f"Ablated_{d}")
    all_models.append(teacher_model)          # reuse; vector does the work
    all_ablation_vectors.append(_normalised_nullspace(teacher_model, d).to(device))

# Add any external student models
for student_run in STUDENT_RUNS:
    stu_model, _, _ = load_model_from_run(student_run, device=device)
    stu_model.eval()
    label = Path(student_run).name if "/" in student_run else student_run
    all_labels.append(label)
    all_models.append(stu_model)
    all_ablation_vectors.append(None)

REFERENCE_IDX = 0   # Healthy is always the reference

print(f"Models to compare ({len(all_labels)} total):")
for i, lbl in enumerate(all_labels):
    tag = " ← reference" if i == REFERENCE_IDX else ""
    print(f"  [{i}] {lbl}{tag}")

## Single-trial analysis

Sample from all models for one `(CUE, COLOR1_DEG, COLOR2_DEG)` input.

In [ ]:
trial_info = build_trial_info(task, cue=CUE, color1_deg=COLOR1_DEG, color2_deg=COLOR2_DEG, num_samples=NUM_SAMPLES)
print(f"Trial: cue={CUE}  color1={COLOR1_DEG:.0f}°  color2={COLOR2_DEG:.0f}°  (same color: {COLOR1_DEG == COLOR2_DEG})")

In [ ]:
all_samples_dicts = []
with torch.no_grad():
    for label, model_inst, abl_vec in zip(all_labels, all_models, all_ablation_vectors):
        _, sd = generate_samples_for_model(model_inst, trial_info, NUM_SAMPLES, device, abl_vec)
        all_samples_dicts.append(sd)
        print(f"  {label}: done  (samples shape {sd['samples'].shape})")

In [ ]:
sw_scores = compute_sw_scores(
    all_labels, all_samples_dicts,
    reference_idx=REFERENCE_IDX,
    student_vs_student=False,
    n_projections=N_SW_PROJECTIONS,
    seed=SEED,
)

# Display as a sorted table
ref_label = all_labels[REFERENCE_IDX]
rows = [{"model": lB if lA == ref_label else lA, "sw_vs_healthy": sw}
        for (lA, lB), sw in sw_scores.items()]
sw_df = pd.DataFrame(rows).sort_values("sw_vs_healthy", ascending=False).reset_index(drop=True)
display(sw_df)

### Training-style multi-panel plot

One row per model. Teacher gets black borders; students/ablations get tab10 colours.

In [ ]:
out_path = str(OUT_DIR / f"cue{CUE}_c1{COLOR1_DEG:.0f}_c2{COLOR2_DEG:.0f}_full.png")
plot_single_trial(
    trial_info=trial_info,
    all_labels=all_labels,
    all_samples_dicts=all_samples_dicts,
    task=task,
    num_timesteps=teacher_model.T,
    out_path=out_path,
    cue=CUE,
    color1_deg=COLOR1_DEG,
    color2_deg=COLOR2_DEG,
    reference_idx=REFERENCE_IDX,
    sw_scores=sw_scores,
)
from IPython.display import Image
Image(out_path)

### Compact overlay: all samples on one plot

Useful for quick visual comparison when there are many models.

In [ ]:
tab10 = plt.get_cmap("tab10")
n = len(all_labels)
ncols = min(n, 5)
nrows = (n + ncols - 1) // ncols + 1  # +1 for the combined panel

fig = plt.figure(figsize=(5 * ncols, 5 * nrows))

# ── Per-model scatter panels ──────────────────────────────────────────────────
student_ci = 0
for i, (label, sd) in enumerate(zip(all_labels, all_samples_dicts)):
    ax = fig.add_subplot(nrows, ncols, i + 1)
    is_ref = (i == REFERENCE_IDX)
    color = "black" if is_ref else tab10(student_ci)
    if not is_ref:
        student_ci += 1

    xy = _samples_to_2d(sd, task)
    ax.scatter(xy[:, 0], xy[:, 1], s=2, alpha=0.5, color=color)
    ax.add_patch(plt.Circle((0, 0), getattr(task.sample_gen, "sample_radius", 1.0), color="red", fill=False))

    sw_val = sw_scores.get((ref_label, label), sw_scores.get((label, ref_label), None))
    sw_str = f"\nSW={sw_val:.4f}" if sw_val is not None else ""
    role = " [teacher]" if is_ref else ""
    ax.set_title(f"{label}{role}{sw_str}", fontsize=9)
    lim = getattr(task.sample_gen, "sample_radius", 1.0) * 1.4
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(2.5 if is_ref else 1.0)

# ── Combined overlay panel (bottom row, full width) ───────────────────────────
ax_all = fig.add_subplot(nrows, 1, nrows)
ax_all.set_title(f"All samples overlaid  (cue={CUE} | c1={COLOR1_DEG:.0f}° | c2={COLOR2_DEG:.0f}°)")

student_ci = 0
draw_order = [i for i in range(n) if i != REFERENCE_IDX] + [REFERENCE_IDX]
for i in draw_order:
    is_ref = (i == REFERENCE_IDX)
    color = "black" if is_ref else tab10(student_ci)
    if not is_ref:
        student_ci += 1
    xy = _samples_to_2d(all_samples_dicts[i], task)
    ax_all.scatter(xy[:, 0], xy[:, 1], s=4 if is_ref else 1,
                   alpha=0.7 if is_ref else 0.15, color=color,
                   label=all_labels[i], zorder=10 if is_ref else 1)

ax_all.add_patch(plt.Circle((0, 0), getattr(task.sample_gen, "sample_radius", 1.0), color="red", fill=False))
ax_all.legend(ncol=min(n, 8), fontsize=8, markerscale=5, loc="upper right")
lim = getattr(task.sample_gen, "sample_radius", 1.0) * 1.4
ax_all.set_xlim(-lim, lim); ax_all.set_ylim(-lim, lim)
ax_all.set_aspect("equal")

fig.tight_layout()
fig.savefig(OUT_DIR / f"cue{CUE}_c1{COLOR1_DEG:.0f}_c2{COLOR2_DEG:.0f}_overlay.png", dpi=150)
plt.show()

---
## Sweep: all 288 inputs

Computes SW(Healthy, Ablated_k) for every (cue, color1, color2) combination.  
Results are saved as `sw_sweep.csv` in `OUT_DIR`.

In [ ]:
ANGLE_STEP   = 30      # degrees; 30 → 288 trials
SWEEP_SUBSET = 0       # how many trials to also save as full plots (0 = none)

trials = generate_sweep_trials(angle_step=ANGLE_STEP)
rng    = np.random.default_rng(SEED)
plot_trial_indices = set(
    rng.choice(len(trials), size=min(SWEEP_SUBSET, len(trials)), replace=False).tolist()
) if SWEEP_SUBSET > 0 else set()

sweep_rows = []
sweep_plot_dir = OUT_DIR / "sweep_plots"
if plot_trial_indices:
    sweep_plot_dir.mkdir(exist_ok=True)

with torch.no_grad():
    for tidx, trial in enumerate(tqdm(trials, desc="Sweep")):
        cue, c1, c2 = trial["cue"], trial["color1_deg"], trial["color2_deg"]

        ti = build_trial_info(task, cue=cue, color1_deg=c1, color2_deg=c2, num_samples=NUM_SAMPLES)

        sds = []
        for model_inst, abl_vec in zip(all_models, all_ablation_vectors):
            _, sd = generate_samples_for_model(model_inst, ti, NUM_SAMPLES, device, abl_vec)
            sds.append(sd)

        sw = compute_sw_scores(
            all_labels, sds,
            reference_idx=REFERENCE_IDX,
            student_vs_student=False,
            n_projections=N_SW_PROJECTIONS,
            seed=SEED + tidx,
        )

        row = {"trial_idx": tidx, "cue": cue, "color1_deg": c1, "color2_deg": c2, "same_color": c1 == c2}
        for (lA, lB), v in sw.items():
            row[f"sw_{lA}_vs_{lB}"] = v
        ref_vs = [v for (lA, lB), v in sw.items() if lA == ref_label or lB == ref_label]
        if len(ref_vs) > 1:
            row["teacher_vs_students_mean"] = float(np.mean(ref_vs))
            row["teacher_vs_students_std"]  = float(np.std(ref_vs))
        sweep_rows.append(row)

        if tidx in plot_trial_indices:
            plot_single_trial(
                trial_info=ti, all_labels=all_labels, all_samples_dicts=sds,
                task=task, num_timesteps=teacher_model.T,
                out_path=str(sweep_plot_dir / f"trial{tidx:04d}_cue{cue}_c1{c1:.0f}_c2{c2:.0f}.png"),
                cue=cue, color1_deg=c1, color2_deg=c2,
                reference_idx=REFERENCE_IDX, sw_scores=sw,
            )

sweep_df = pd.DataFrame(sweep_rows)
sweep_csv = OUT_DIR / "sw_sweep.csv"
sweep_df.to_csv(sweep_csv, index=False)
print(f"Sweep complete. {len(sweep_rows)} trials. Saved: {sweep_csv}")
display(sweep_df.head())

### Sweep results: SW per ablation direction

Bar chart of mean SW across all 288 trials, broken down by ablation direction.  
Run the cell below after the sweep cell has finished.

In [ ]:
# Load from CSV if re-running analysis on existing results
# sweep_df = pd.read_csv(OUT_DIR / "sw_sweep.csv")

sw_cols = [c for c in sweep_df.columns if c.startswith("sw_Healthy_vs_")]
ablation_labels = [c.replace("sw_Healthy_vs_", "") for c in sw_cols]

means_all   = [sweep_df[c].mean()                          for c in sw_cols]
means_same  = [sweep_df.loc[sweep_df["same_color"], c].mean()  for c in sw_cols]
means_diff  = [sweep_df.loc[~sweep_df["same_color"], c].mean() for c in sw_cols]

x = np.arange(len(ablation_labels))
w = 0.25

fig, ax = plt.subplots(figsize=(max(8, len(ablation_labels) * 0.7), 4))
ax.bar(x - w, means_all,  width=w, label="All trials",        alpha=0.85)
ax.bar(x,     means_same, width=w, label="Same color",        alpha=0.85)
ax.bar(x + w, means_diff, width=w, label="Different colors",  alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(ablation_labels, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Mean sliced-Wasserstein (vs Healthy)")
ax.set_title("Effect of ablating each nullspace direction on output distribution")
ax.legend()
fig.tight_layout()
fig.savefig(OUT_DIR / "sw_per_ablation_direction_bar.png", dpi=150)
plt.show()

# Summary table
summary = pd.DataFrame({
    "ablation": ablation_labels,
    "mean_all": means_all, "mean_same_color": means_same, "mean_diff_color": means_diff,
}).sort_values("mean_all", ascending=False)
display(summary)

In [ ]:
### Heatmap: SW as a function of (color1, color2) for each ablation direction
# One subplot per direction; rows = color1, cols = color2; cued item = 1

HEATMAP_CUE = 1   # which cue to plot (1 or 2)

angles = list(range(0, 360, ANGLE_STEP))
n_ang  = len(angles)
angle_to_idx = {a: i for i, a in enumerate(angles)}

ncols_h = min(4, len(sw_cols))
nrows_h = (len(sw_cols) + ncols_h - 1) // ncols_h

fig, axes = plt.subplots(nrows_h, ncols_h, figsize=(5 * ncols_h, 4.5 * nrows_h))
axes_flat = np.array(axes).flatten() if len(sw_cols) > 1 else [axes]

sub_df = sweep_df[sweep_df["cue"] == HEATMAP_CUE]
vmin = sweep_df[[c for c in sw_cols]].min().min()
vmax = sweep_df[[c for c in sw_cols]].max().max()

for ax, col in zip(axes_flat, sw_cols):
    mat = np.full((n_ang, n_ang), np.nan)
    for _, row in sub_df.iterrows():
        r = angle_to_idx.get(int(row["color1_deg"]))
        c = angle_to_idx.get(int(row["color2_deg"]))
        if r is not None and c is not None:
            mat[r, c] = row[col]
    im = ax.imshow(mat, origin="lower", vmin=vmin, vmax=vmax, cmap="viridis", aspect="equal")
    ax.set_title(col.replace("sw_Healthy_vs_", "Ablated "), fontsize=9)
    ax.set_xlabel("color2 (deg)"); ax.set_ylabel("color1 (deg)")
    ax.set_xticks(range(n_ang)); ax.set_xticklabels(angles, fontsize=6, rotation=45)
    ax.set_yticks(range(n_ang)); ax.set_yticklabels(angles, fontsize=6)
    plt.colorbar(im, ax=ax, shrink=0.7)

for ax in axes_flat[len(sw_cols):]:
    ax.axis("off")

fig.suptitle(f"SW(Healthy, Ablated) as fn of (c1, c2) — cue={HEATMAP_CUE}", fontsize=11)
fig.tight_layout()
fig.savefig(OUT_DIR / f"sw_heatmap_cue{HEATMAP_CUE}.png", dpi=150)
plt.show()